## Setup

Run the cells below to install dependencies and configure the language subset you want to fine-tune on.

In [ ]:
!pip install -q seqeval #accelerate datasets evaluate seqeval transformers

In [ ]:
import numpy as np
import torch
from datasets import get_dataset_config_names, load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
import evaluate

model_checkpoint = "castorini/afriberta_small"
available_languages = get_dataset_config_names("masakhane/masakhapos")
print(f"{len(available_languages)} languages available. Examples: {available_languages[:10]}")
language_code = "hau"  # Change this to another MasakhaNER2 language code when needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from datasets import DatasetDict, concatenate_datasets

language_code = "multilingual"

raw_datasets_per_lang = [load_dataset("masakhane/masakhapos", lang) for lang in available_languages]

all_splits = set()
for ds in raw_datasets_per_lang:
    all_splits.update(ds.keys())

raw_datasets = DatasetDict(
    {
        split: concatenate_datasets([ds[split] for ds in raw_datasets_per_lang if split in ds])
        for split in sorted(all_splits)
    }
)
print(raw_datasets)

label_list = raw_datasets["train"].features["upos"].feature.names
num_labels = len(label_list)
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

split_names = list(raw_datasets.keys())
print(f"Available splits: {split_names}")
eval_split = "validation" if "validation" in raw_datasets else "test"

sample = raw_datasets["train"][0]
print("Sample tokens:", sample["tokens"])
print("Sample tags:", [label_list[tag] for tag in sample["upos"]])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
label_all_tokens = False  # Set to True to propagate labels to all wordpieces

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for batch_index, label_sequence in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_sequence[word_idx])
            else:
                label_ids.append(label_sequence[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
tokenized_datasets

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        filtered_preds = []
        filtered_labels = []
        for pred, lbl in zip(prediction, label):
            if lbl != -100:
                filtered_preds.append(label_list[pred])
                filtered_labels.append(label_list[lbl])
        true_predictions.append(filtered_preds)
        true_labels.append(filtered_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model

In [ ]:
model_name = model_checkpoint.split("/")[-1]
run_name = f"{model_name}-{language_code}-ner"

training_args = TrainingArguments(
    output_dir=f"results/{run_name}",
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    gradient_accumulation_steps=1,
    # fp16=torch.cuda.is_available(),
    report_to="none",
)
training_args

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets[eval_split],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
metrics = trainer.evaluate(tokenized_datasets["test"])
metrics

In [ ]:
save_dir = f"results/{run_name}/final-model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
save_dir

## Next Steps

- Adjust hyperparameters to better match dataset size and hardware.
- Enable logging integrations such as Weights & Biases by updating TrainingArguments.report_to.
- Push the fine-tuned weights to the Hugging Face Hub with trainer.push_to_hub() if you want to share the model.